# Модель A (мин.): классификатор дизайна исследования

LR-голова на замороженных эмбеддингах. Цель — **забанковать требование «обучено в Yandex Cloud»** (plan §7).
Запуск: DataSphere → project → Run All. Минуты на CPU. Датасет ниже — крошечный демо-сэмпл;
заменить на human-checked gold (50–100) из `data/`.

In [ ]:
# В DataSphere раскомментируйте при необходимости:
# %pip install -q sentence-transformers scikit-learn joblib

In [ ]:
# Демо-сэмпл (abstract, design). Заменить на gold-набор.
data = [
    ("Randomized controlled trial comparing drug A vs placebo, double-blind.", "rct"),
    ("We randomly assigned 200 patients to two arms in a phase III trial.", "rct"),
    ("Retrospective cohort study of registry data over 10 years.", "observational"),
    ("A prospective observational cohort followed for survival outcomes.", "observational"),
    ("This systematic review and meta-analysis pooled 12 studies.", "review"),
    ("Narrative review of immunotherapy in melanoma.", "review"),
    ("Case report of a rare adverse event in one patient.", "case_report"),
    ("We describe a single case of unexpected response.", "case_report"),
]
texts = [t for t, _ in data]
labels = [y for _, y in data]
len(texts)

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("intfloat/multilingual-e5-small")
X = encoder.encode(texts, normalize_embeddings=True)
X.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = LogisticRegression(max_iter=1000)
clf.fit(X, labels)
# Демо-набор крошечный → метрика на train. На gold — честный hold-out per-class F1.
print(classification_report(labels, clf.predict(X)))

In [ ]:
import joblib

joblib.dump(clf, "design_clf.joblib")
# Чекпойнт → Object Storage (бакет проекта). Пример (boto3, S3-совместимый эндпоинт YC):
# import boto3
# s3 = boto3.client('s3', endpoint_url='https://storage.yandexcloud.net')
# s3.upload_file('design_clf.joblib', '<bucket>', 'models/design_clf.joblib')